In [1]:
"""
Personalized training  +  FEATURE METHOD 2: concatenate-after-encoder.

Each cycle is time-normalized to TARGET_LEN points and fed to an LSTM/GRU/CNN
encoder. Two scalar gait features (cadence, duration variability) enter through
a SECOND input branch and are concatenated onto the encoder output before the
final dense layer (Keras functional API).

Set USE_FEATURES = False to get the sequence-only baseline (single input).

Leak-safety: duration variability is computed on the TRAIN cycles of each
subject only; cadence is per-cycle; scalars standardized with train statistics.

SESSION SPLIT WITH GUARD BANDS (replaces the random 80/20 split):
Within each subject, the cycles of each gait class are put back in their
original temporal order (by cycle id) and grouped into consecutive SESSIONS of
SESSION_LEN cycles. Between every two sessions, GUARD_LEN cycles are discarded
as a guard band. Whole sessions are then randomly assigned to train or test so
that about TEST_FRACTION of the sessions are used for testing. Because every
session boundary has a guard band, no training cycle is ever adjacent to a test
cycle, and only the guard cycles are discarded.

REPEATS: the whole split + training is repeated N_REPEATS times, each time with
a different random choice of test sessions. For a given repeat and subject the
same split is used for all three sensor configurations, and it is identical
between the USE_FEATURES=True and USE_FEATURES=False runs, so the two runs can
be compared pair by pair (see compare_featconcat_vs_seqonly.py).
"""
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, GRU, Conv1D, GlobalAveragePooling1D,
                                     Dense, Dropout, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# Configuration
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100
SAMPLING_RATE = 100.0
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']

USE_FEATURES = True        # <-- toggle for ablation

IMU_COLUMNS = [
    'IMU101_v0', 'IMU101_v1', 'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
]
PRESSURE_COLUMNS = [
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7',
    'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]
CONFIGS = {
    'IMU':          IMU_COLUMNS,
    'Pressure':     PRESSURE_COLUMNS,
    'IMU+Pressure': IMU_COLUMNS + PRESSURE_COLUMNS,
}

UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32
MODEL_TYPES = ['lstm', 'gru', 'cnn']

# Session split settings
SESSION_LEN = 20     # consecutive cycles per session
GUARD_LEN = 6        # cycles discarded between consecutive sessions
TEST_FRACTION = 0.2  # share of sessions (per class) assigned to testing
SPLIT_SEED = 42      # base seed; repeat r uses its own seed derived from this
N_REPEATS = 5        # number of different random train/test session assignments

RESULTS_DIR = 'gait_hardware_paper_results'
tag = 'featconcat' if USE_FEATURES else 'seqonly_concat'
RESULTS_CSV = os.path.join(RESULTS_DIR, f'personalized_{tag}_sessions_results.csv')        # every repeat
SUMMARY_CSV = os.path.join(RESULTS_DIR, f'personalized_{tag}_sessions_summary.csv')        # mean over repeats


def get_subject_id(file_path):
    return os.path.relpath(file_path, ROOT_DIR).split(os.sep)[0]


def resample_cycle(features, target_len=TARGET_LEN):
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_all_raw():
    raw = []
    if not os.path.exists(ROOT_DIR):
        print(f"ROOT_DIR not found: {ROOT_DIR}")
        return raw
    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                fp = os.path.join(root, filename)
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cid = raw_id_part.split('_')[-1]
                if label not in VALID_LABELS or not cid.isdigit():
                    continue
                try:
                    df = pd.read_csv(fp)
                except Exception:
                    continue
                raw.append((df, label, get_subject_id(fp), int(cid)))
    return raw


def prepare_subject_matrix(subject_rows, feature_cols):
    X, y, nrows, cids = [], [], [], []
    for df, label, _, cid in subject_rows:
        if any(c not in df.columns for c in feature_cols):
            continue
        vals = df[feature_cols].values
        if vals.shape[0] < 2 or vals.shape[1] == 0:
            continue
        nrows.append(vals.shape[0])
        X.append(resample_cycle(vals))
        y.append(label)
        cids.append(cid)
    if len(X) == 0:
        return None, None, None, None
    return np.array(X), np.array(y), np.array(nrows), np.array(cids)


def make_sessions(ordered_idx, session_len, guard_len):
    """Cut temporally ordered indices into sessions separated by guard bands.
    Layout: [session][guard][session][guard]...[session].
    A final partial session is kept if it has at least half of session_len,
    otherwise its cycles are discarded with the guard cycles."""
    sessions, guard = [], []
    i, n = 0, len(ordered_idx)
    while i < n:
        sess = ordered_idx[i:i + session_len]
        if len(sess) == session_len or len(sess) >= session_len / 2:
            sessions.append(sess)
        else:
            guard.extend(sess)
        i += session_len
        guard.extend(ordered_idx[i:i + guard_len])   # guard band after the session
        i += guard_len
    return sessions, guard


def session_guard_split(y_int, cycle_ids, session_len, guard_len, test_fraction, rng):
    """For each class: order by cycle id, build sessions with guard bands,
    randomly assign whole sessions to test (about test_fraction) and train."""
    tr, te, guard = [], [], []
    n_tr_sess, n_te_sess = 0, 0
    for c in np.unique(y_int):
        idx = np.where(y_int == c)[0]
        idx = idx[np.argsort(cycle_ids[idx], kind='stable')]   # temporal order
        sessions, g = make_sessions(idx, session_len, guard_len)
        guard.extend(g)
        k = len(sessions)
        n_test = max(1, int(round(test_fraction * k)))
        test_ids = set(rng.choice(k, size=n_test, replace=False).tolist())
        for j, sess in enumerate(sessions):
            if j in test_ids:
                te.extend(sess); n_te_sess += 1
            else:
                tr.extend(sess); n_tr_sess += 1
    return (np.array(tr, dtype=int), np.array(te, dtype=int),
            np.array(guard, dtype=int), n_tr_sess, n_te_sess)


def build_model(model_type, seq_len, num_features, num_scalars, num_classes, use_features):
    """Functional model. If use_features, a second scalar input is concatenated
    after the sequence encoder; otherwise it's a single-input model."""
    seq_in = Input(shape=(seq_len, num_features), name='sequence')
    if model_type == 'lstm':
        x = LSTM(UNITS, name='LSTM')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
    elif model_type == 'gru':
        x = GRU(UNITS, name='GRU')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
    elif model_type == 'cnn':
        x = Conv1D(64, 5, activation='relu')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
        x = Conv1D(64, 5, activation='relu')(x)
        x = GlobalAveragePooling1D()(x)
        x = Dropout(DROPOUT_RATE)(x)
    else:
        raise ValueError(model_type)

    if use_features:
        scal_in = Input(shape=(num_scalars,), name='scalars')
        x = Concatenate()([x, scal_in])
        out = Dense(num_classes, activation='softmax')(x)
        model = Model(inputs=[seq_in, scal_in], outputs=out)
    else:
        out = Dense(num_classes, activation='softmax')(x)
        model = Model(inputs=seq_in, outputs=out)

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def run():
    raw = load_all_raw()
    if not raw:
        print("No cycles loaded.")
        return
    subjects = sorted({r[2] for r in raw})
    print(f"Detected {len(subjects)} subjects: {subjects}")
    print(f"USE_FEATURES = {USE_FEATURES}\n")

    rows = []
    for rep_i in range(N_REPEATS):
        print(f"\n{'#'*55}\n### Repeat {rep_i + 1} of {N_REPEATS}\n{'#'*55}")
        for subj_i, subj in enumerate(subjects):
            subj_rows = [r for r in raw if r[2] == subj]
            print(f"\n{'='*55}\n=== Subject: {subj} ({len(subj_rows)} cycles) ===\n{'='*55}")

            for cfg_name, cols in CONFIGS.items():
                X, y_str, nrows, cids = prepare_subject_matrix(subj_rows, cols)
                if X is None:
                    print(f"  [{cfg_name}] no usable cycles, skipping")
                    continue

                le = LabelEncoder()
                y_int = le.fit_transform(y_str)
                num_classes = len(le.classes_)
                y_cat = to_categorical(y_int, num_classes=num_classes)

                n = X.shape[0]
                uniq, counts = np.unique(y_int, return_counts=True)
                min_needed = 3 * SESSION_LEN + 2 * GUARD_LEN   # at least 3 sessions per class
                if counts.min() < min_needed or num_classes < 2:
                    print(f"  [{cfg_name}] too few cycles per class for sessions "
                          f"(min/class={counts.min()}, need {min_needed}), skipping")
                    continue

                # sanity check: cycle ids must be unique within each class,
                # otherwise the temporal order is ambiguous
                for c in uniq:
                    ids_c = cids[y_int == c]
                    if len(np.unique(ids_c)) != len(ids_c):
                        print(f"  WARNING [{cfg_name}] duplicate cycle ids in class "
                              f"{le.classes_[c]} -- temporal order may be unreliable")

                # same seed for every config and for both USE_FEATURES runs -> same split
                rng = np.random.default_rng([SPLIT_SEED, rep_i, subj_i])
                tr_idx, te_idx, guard_idx, n_tr_sess, n_te_sess = session_guard_split(
                    y_int, cids, SESSION_LEN, GUARD_LEN, TEST_FRACTION, rng)
                print(f"  [{cfg_name}] session split: train={len(tr_idx)} ({n_tr_sess} sessions) "
                      f"test={len(te_idx)} ({n_te_sess} sessions) guard(discarded)={len(guard_idx)}")

                cadence = SAMPLING_RATE / nrows
                dur_std_train = float(np.std(nrows[tr_idx]))
                scalar = np.column_stack([cadence, np.full(n, dur_std_train)])
                num_scalars = scalar.shape[1]

                if USE_FEATURES:
                    scaler = StandardScaler().fit(scalar[tr_idx])
                    scalar = scaler.transform(scalar)

                seq_len = X.shape[1]
                num_features = X.shape[2]
                Xtr_s, Xte_s = X[tr_idx], X[te_idx]
                sctr, scte = scalar[tr_idx], scalar[te_idx]
                ytr, yte = y_cat[tr_idx], y_cat[te_idx]
                yte_int = y_int[te_idx]

                for mtype in MODEL_TYPES:
                    tf.keras.backend.clear_session()
                    model = build_model(mtype, seq_len, num_features, num_scalars,
                                        num_classes, USE_FEATURES)
                    train_x = [Xtr_s, sctr] if USE_FEATURES else Xtr_s
                    test_x = [Xte_s, scte] if USE_FEATURES else Xte_s
                    model.fit(train_x, ytr, epochs=EPOCHS, batch_size=BATCH_SIZE,
                              callbacks=[EarlyStopping(monitor='loss', patience=5,
                                                       restore_best_weights=True, verbose=0)],
                              verbose=0)
                    pred = np.argmax(model.predict(test_x, verbose=0), axis=1)
                    acc = accuracy_score(yte_int, pred)
                    print(f"  [{cfg_name:12s} {mtype.upper():4s}] "
                          f"n_test={len(yte_int):3d}  acc={acc:.4f}")
                    rows.append({
                        'repeat': rep_i + 1,
                        'subject': subj, 'config': cfg_name, 'model': mtype,
                        'use_features': USE_FEATURES, 'feature_method': 'concat',
                        'n_cycles': n, 'n_train': len(tr_idx), 'n_test': len(yte_int),
                        'n_guard': len(guard_idx), 'n_train_sessions': n_tr_sess,
                        'n_test_sessions': n_te_sess, 'n_classes': num_classes,
                        'accuracy': round(acc, 4),
                        'macro_f1': round(f1_score(yte_int, pred, average='macro', zero_division=0), 4),
                        'weighted_f1': round(f1_score(yte_int, pred, average='weighted', zero_division=0), 4),
                    })

    if not rows:
        print("\nNo results produced.")
        return
    os.makedirs(RESULTS_DIR, exist_ok=True)
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    print(f"\n\nWrote {len(df)} rows (all repeats) to {RESULTS_CSV}")

    summary = (df.groupby(['subject', 'config', 'model'])
                 .agg(accuracy=('accuracy', 'mean'),
                      accuracy_std_over_repeats=('accuracy', 'std'),
                      macro_f1=('macro_f1', 'mean'),
                      weighted_f1=('weighted_f1', 'mean'),
                      n_repeats=('repeat', 'nunique'))
                 .round(4).reset_index())
    summary['use_features'] = USE_FEATURES
    summary.to_csv(SUMMARY_CSV, index=False)
    print(f"Wrote mean over {N_REPEATS} repeats to {SUMMARY_CSV}")
    with pd.option_context('display.max_rows', None, 'display.width', 120):
        print(summary.pivot_table(index='subject', columns=['config', 'model'], values='accuracy'))


if __name__ == '__main__':
    tf.get_logger().setLevel('ERROR')
    run()

2026-10-05 16:12:22.729942: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791231142.861359   85390 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791231142.899531   85390 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791231143.207929   85390 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791231143.207948   85390 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791231143.207952   85390 computation_placer.cc:177] computation placer alr

Detected 10 subjects: ['Andy_Dynamic', 'Ankan_Dynamic', 'David_Dynamic', 'Hrithik_Dynamic', 'JJ_Dynamic', 'Mustafa_Dynamic', 'Rezoan_Dynamic', 'Sudipta_Dynamic', 'Tianjun_Dynamic', 'Zongwei_Dynamic']
USE_FEATURES = True


#######################################################
### Repeat 1 of 5
#######################################################

=== Subject: Andy_Dynamic (424 cycles) ===
  [IMU] session split: train=250 (13 sessions) test=80 (4 sessions) guard(discarded)=94


I0000 00:00:1791231246.124691   85390 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46677 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1791231246.130030   85390 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 46575 MB memory:  -> device: 1, name: NVIDIA RTX A6000, pci bus id: 0000:25:00.0, compute capability: 8.6
I0000 00:00:1791231246.131480   85390 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 46677 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1791231246.132918   85390 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 46677 MB memory:  -> device: 3, name: NVIDIA RTX A6000, pci bus id: 0000:c1:00.0, compute capability: 8.6
I0000 00:00:1791231249.187432   86219 cuda_dnn.cc:529] Loaded cuDNN version 90701


  [IMU          LSTM] n_test= 80  acc=0.9625
  [IMU          GRU ] n_test= 80  acc=0.8750


I0000 00:00:1791231263.362626   86216 service.cc:152] XLA service 0x7fd16402bf20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791231263.362705   86216 service.cc:160]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1791231263.362708   86216 service.cc:160]   StreamExecutor device (1): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1791231263.362710   86216 service.cc:160]   StreamExecutor device (2): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1791231263.362712   86216 service.cc:160]   StreamExecutor device (3): NVIDIA RTX A6000, Compute Capability 8.6
2026-10-05 16:14:23.410561: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791231265.899220   86216 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the pro

  [IMU          CNN ] n_test= 80  acc=0.9750
  [Pressure] session split: train=250 (13 sessions) test=80 (4 sessions) guard(discarded)=94
  [Pressure     LSTM] n_test= 80  acc=0.9625
  [Pressure     GRU ] n_test= 80  acc=0.9875
  [Pressure     CNN ] n_test= 80  acc=0.9750
  [IMU+Pressure] session split: train=250 (13 sessions) test=80 (4 sessions) guard(discarded)=94
  [IMU+Pressure LSTM] n_test= 80  acc=0.9375
  [IMU+Pressure GRU ] n_test= 80  acc=0.9750
  [IMU+Pressure CNN ] n_test= 80  acc=0.9500

=== Subject: Ankan_Dynamic (528 cycles) ===
  [IMU] session split: train=320 (16 sessions) test=80 (4 sessions) guard(discarded)=128
  [IMU          LSTM] n_test= 80  acc=0.8750
  [IMU          GRU ] n_test= 80  acc=0.8875
  [IMU          CNN ] n_test= 80  acc=0.9125
  [Pressure] session split: train=320 (16 sessions) test=80 (4 sessions) guard(discarded)=128
  [Pressure     LSTM] n_test= 80  acc=0.8750
  [Pressure     GRU ] n_test= 80  acc=0.7750
  [Pressure     CNN ] n_test= 80  acc=0.95